# 04 - Paper parity: where every source figure stands

**What this notebook is for.** The replication contract: every figure and table from the two
source papers is either replicated, substituted with the difference documented, queued with
its cost, or declared out of scope with the reason. Canonical inventory:
`notes/plot_parity.md` (rendered below); this notebook also carries the results that live
nowhere else: the logit-lens split verdict and the preference (Elo) experiment.

**Key concepts.**
- *Battery*: our name for the scenario test set — the Anthropic paper's 12
  implicit-emotion prompts (its Table 2), plus our own held-out set of 12.
- *Logit lens*: projecting an internal direction through the model's output vocabulary matrix
  to see which tokens it up-weights.
- *Elo rating*: a single preference-strength score per activity, fitted (Bradley-Terry) from
  all pairwise A/B choices so that Elo gaps predict win probabilities.
- *Substitution*: same analysis, different tool (for example t-SNE for UMAP), always with the
  difference stated.

**Index.**
1. The inventory (both papers)
2. Logit lens (paper Table 1): instruct fails, base partially reproduces
3. Preferences (paper Figure 4, left half): probe activations vs revealed preference
4. Steering (paper Figure 4, right half): the causal test

## 1. The inventory

### Anthropic paper

| Item | What it shows | Status | Where / why |
|---|---|---|---|
| Figure 1 | Top-activating dataset snippets per emotion vector, external corpora | QUEUED (scaled down) | Needs a corpus sweep (LMSYS/Pile samples) with per-token projection; ~half a pod day; not gate-critical |
| Table 1 | Logit-lens top/bottom tokens per emotion vector | SPLIT: base partial, instruct negative | Section 2 below: base vectors at layer 33 show affective token neighborhoods for about half the emotions; instruct vectors show none at layers 33 or 57. Final-norm scaling applied, softcapping ignored |
| Figure 2 | Probe x scenario cosine matrix, strong diagonal | DONE | notebooks/03, sections 1 and 3 (dual-model); our diagonals are weaker, which is a finding (TREE Q1.H2) |
| Table 2 | The 12 implicit-emotion scenarios | DONE | Used verbatim, src/emotion_vectors/probe_prompts.py |
| Figure 3 | Numerical-intensity template curves | DONE | notebooks/03, section 2 (dual-model): instruct tracks 11/11 registered directions, base 7/11 |
| Figure 4 | Activity-preference Elo + steering shifts | REPRODUCES (after instrument fix) | Sections 3-4 below (TREE Q1.H3 [supported]): probe-Elo max abs r 0.70 at layer 33 (paper 0.71-0.74), Elo split 1727 vs -578; steering valence-sign test 11/12 and 10/12, dose-responsive. The earlier negative verdicts were artifacts of a padding bug (Q1.H3.E4); plain format remains a dead instrument. Below-bar remainder: delta-vs-r coupling 0.32 (paper 0.85) |
| Figure 5 | Pairwise cosine similarity, clustered | DONE | notebooks/02, section 3 |
| Figure 6 | UMAP of k-means emotion clusters | SUB, DONE | notebooks/archive/09: t-SNE embedding instead of UMAP (dependency), identical k-means k=10; clusters interpretable (joy/hope family, calm/content family), matching the paper's qualitative result |
| Figure 7 | PC1/PC2 loading bars per emotion | DONE | notebooks/02, section 5: each model in its own valence-best/arousal-best component plane |
| Figure 8 | PC1/PC2 vs human valence/arousal ratings | SUB | We correlate against the NRC VAD lexicon (the replication's instrument), not Russell's 45-emotion ratings; documented in TREE Q1.H1.C1 |
| Figure 9 | Representational similarity across layers | DONE | notebooks/02, section 4 |
| Neutral-PC projection (methods) | Confound removal before probe use | DONE (late) | Missing from the reference code and our pipeline until 2026-07-21; E7 implements it |
| Appendix: token-level activation localization | Vectors activate on emotion-relevant story spans | QUEUED | Requires per-token projection; shared infrastructure with Q3 |
| Overview panels: reward-hacking steering | Steering shifts misalignment rates | OUT | Production alignment evals and steering infra; not reproducible here |

### Open replication (sinievanderben/emotion_experiment)

| Item | What it shows | Status | Where / why |
|---|---|---|---|
| fig1_cosine_similarity | Contrast-vector cosine heatmap | DONE | notebooks/02, section 3 |
| fig2_pca | PCA scatter + valence/arousal panels | DONE | notebooks/02, sections 1-2 |
| fig3_umap | UMAP colored by k-means cluster | SUB, DONE | Same t-SNE substitution as the paper's Figure 6; notebooks/archive/09 |
| fig_valence/arousal_trajectory | PC-correlation across layers, two models | DONE + extended | notebooks/02 (base); our base-vs-instruct comparison (results/emotion_geometry_correlations*.json) is the same plot family with a new finding (valence demotion, TREE Q1.H1.C2) |
| fig_cka (centered kernel alignment) | Cross-layer representation similarity | SUB | We use representational similarity analysis (correlation of pairwise-cosine structures) instead of CKA; same question, different similarity index; notebooks/02 section 4 |
| analyze_story_conditions | Same model, vectors from different story corpora | DONE | Our E5 comparison: 4B-corpus vs self-generated probes (notebooks/archive/07) |
| visualize_token_activations | Per-token projection along a sentence | QUEUED | Becomes Q3's core infrastructure (per-token trajectories) |

### Beyond the main results: the complete census

The Anthropic paper contains **86 numbered figures and 16 tables** in total; the table above
covers the main results only. The complete item-by-item census (audited 2026-07-22) lives in
`notes/plot_parity.md`. Summary: ~11 done or substituted, ~74 queued (appendix variants of one
per-token infrastructure, Gemma analogues of transcript illustrations, and base-vs-instruct
proxies for the post-training panels), ~15 out — all on a single blocker class (blackmail /
reward-hacking / sycophancy rollouts and on-policy Claude transcript corpora, e.g. Figures
26-35, 66-68, 76-79). Top queued items by value: the steering delta-log-prob test (Figures
52-53, the causal leg we have never run), the post-training proxy panels (Figures 36-39, 84,
Table 16 — the paper's own quantities for our instruction-tuning finding C2), and self- vs
other-speaker structure (Figures 17-19, 59).

## Maintenance

Update this table whenever a QUEUED item lands or a new figure appears in
either source. The notebook restyle (plotly, skimmable cells) references this
inventory so each notebook states which paper figure it corresponds to.


## 2. Logit lens (paper Table 1): instruct fails, base partially reproduces

In [1]:
# this cell renders the logit-lens token tables for both models: instruct (fails) then base (partial)
import json
from pathlib import Path

import plotly.graph_objects as go

ROOT = Path("..")
STRONG_BASE = [
    "happy",
    "proud",
    "desperate",
    "angry",
    "guilty",
]  # affective neighborhoods, judged by eye


def lens_table(result_path: Path, model_label: str) -> None:
    lens = json.loads(result_path.read_text())
    rows = [(e, ", ".join(t["up"]), ", ".join(t["down"])) for e, t in lens["table"].items()]
    fig = go.Figure(
        go.Table(
            header=dict(
                values=["emotion", "top up-weighted tokens", "top down-weighted tokens"],
                align="left",
            ),
            cells=dict(values=list(zip(*rows)), align="left", height=26),
        )
    )
    fig.update_layout(
        title=(
            f"Logit lens, {model_label}, layer {lens['layer']} ({lens['note']})"
            "<br><sup>maps to: Anthropic Table 1</sup>"
        ),
        height=470,
        margin=dict(t=60, b=10),
    )
    fig.show()


lens_table(ROOT / "results/logit_lens_it_L57.json", "gemma-4-31b-it")
lens_table(ROOT / "results/logit_lens_base_L33.json", "gemma-4-31b (base)")
print(
    "verdict, gemma-4-31b-it: no emotion-word neighborhoods at layer 33 or 57; Table 1 does not reproduce"
)
print(
    f"verdict, gemma-4-31b (base): affective neighborhoods for {len(STRONG_BASE)}/12 at layer 33 ({', '.join(STRONG_BASE)}); valence-consistent down-lists for most others"
)

verdict, gemma-4-31b-it: no emotion-word neighborhoods at layer 33 or 57; Table 1 does not reproduce
verdict, gemma-4-31b (base): affective neighborhoods for 5/12 at layer 33 (happy, proud, desperate, angry, guilty); valence-consistent down-lists for most others


<details><summary><b>How to read these tables</b></summary>

The paper's Table 1 shows each emotion vector up-weighting related words (sad toward grief, tears). Both tables apply the final-normalization scaling (Gemma's 1+w convention); the documented simplification is that logit softcapping is ignored.

- **Instruct (top table)**: unrelated fragments everywhere, at both tested layers (33 shown in `results/logit_lens_it_L33_normed.json`, 57 above). A robust negative.
- **Base (bottom table)**: clear affective neighborhoods for about half the emotions at layer 33 (happy toward delightful/wonderful, angry toward vicious/angrily, desperate toward misery/wretched, guilty toward conceal/incriminating), and valence-consistent down-lists for most others (sad down-weights charming/fabulous, calm down-weights brutal/vicious). Layer 57 (`results/logit_lens_base_L57.json`) is similar but noisier. A partial positive, judged qualitatively, no registered quantitative bar.

The split matters: the same extraction pipeline yields vocabulary-aligned directions on the base model and junk on the instruct model. This is the third independent signature (with the probe-battery failure and the valence demotion, TREE Q1.H1.C2) that instruction tuning buries the affect representation under non-affective structure.

</details>

## 3. Preferences (paper Figure 4, left half): probe activations vs revealed preference

The paper's Figure 4 shows (left) per-emotion probe correlations with activity-preference
Elo ratings and (right) steering shifting those preferences. We reproduce the left half:
Elo from Bradley-Terry over all ordered activity pairs (TREE Q1.H3), on our self-authored
64-activity list (the paper's is unpublished; ours follows its 8 named categories). The
steering half is section 4. All data in sections 3-4 comes from the padding-FIXED collections (TREE Q1.H3.E4); the compromised originals are archived in the results history and on the dataset card. Both prompt formats shown: the paper's exact plain
format, and the model's own chat template (the E4 lesson: plain formatting is
out-of-distribution for an instruct model).

In [2]:
# this cell renders, per collected arm, the paper's Figure 4 left half: probe-Elo bars + best-probe scatter
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

ARMS = [
    ("plain (paper's exact format)", ROOT / "results/preferences_it_fixed"),
    ("chat template", ROOT / "results/preferences_it_chat_fixed"),
]
CATEGORY_COLORS = {
    "helpful": "#2ca02c",
    "engaging": "#17becf",
    "social": "#1f77b4",
    "self_curiosity": "#9467bd",
    "neutral": "#7f7f7f",
    "aversive": "#8c564b",
    "misaligned": "#ff7f0e",
    "unsafe": "#d62728",
}

for arm_label, arm_dir in ARMS:
    scores_file = arm_dir / "scores.json"
    if not scores_file.exists():
        print(f"[{arm_label}] not collected/scored yet — skipped")
        continue
    s = json.loads(scores_file.read_text())
    best = next(b for b in s["probe_elo_by_layer"] if b["layer"] == s["p2_best_layer"])
    r = np.array(best["per_probe_r"])
    order = np.argsort(r)
    names = [s["probe_emotions_matched"][i] for i in order]
    elo_vals = np.array([a["elo"] for a in s["elo_per_activity"]])
    cats = [a["category"] for a in s["elo_per_activity"]]
    act = np.array(best["best_probe_activation_per_activity"])

    fig = make_subplots(
        rows=1,
        cols=2,
        column_widths=[0.55, 0.45],
        horizontal_spacing=0.09,
        subplot_titles=(
            f"per-probe correlation with Elo, layer {best['layer']} (sorted)",
            f"best probe '{s['p2_best_probe_emotion']}' vs Elo (r={best['max_abs_r']})",
        ),
    )
    fig.add_bar(
        x=list(range(len(r))), y=r[order], marker_color="#4878a8", row=1, col=1, showlegend=False
    )
    fig.update_xaxes(
        tickvals=list(range(0, len(r), 8)),
        ticktext=[names[i] for i in range(0, len(r), 8)],
        tickangle=45,
        tickfont=dict(size=9),
        row=1,
        col=1,
    )
    fig.update_yaxes(title_text="Pearson r with Elo", row=1, col=1)
    for cat in CATEGORY_COLORS:
        idx = [i for i, c in enumerate(cats) if c == cat]
        fig.add_scatter(
            x=act[idx],
            y=elo_vals[idx],
            mode="markers",
            name=cat,
            marker=dict(color=CATEGORY_COLORS[cat], size=9),
            row=1,
            col=2,
        )
    fig.update_xaxes(title_text="probe activation (cosine)", row=1, col=2)
    fig.update_yaxes(title_text="Elo rating", row=1, col=2)
    fig.update_layout(
        title=(
            f"Preference Elo vs emotion probes, gemma-4-31b-it, {arm_label}"
            "<br><sup>maps to: Anthropic Figure 4, top-left panels "
            "(steering half is Q1.H3.E2, gated; activity list is our documented substitution)</sup>"
        ),
        height=480,
        legend=dict(orientation="h", y=-0.35),
        margin=dict(t=80, b=10),
    )
    fig.show()
    print(
        f"[{arm_label}] P1 {'PASS' if s['p1_pass'] else 'FAIL'} "
        f"(positive {s['p1_positive_mean']:.0f} vs negative {s['p1_negative_mean']:.0f}); "
        f"P2 {'PASS' if s['p2_pass'] else 'FAIL'} "
        f"(max |r|={s['p2_max_abs_r']} bar 0.5, organization r={s['p2_valence_organization_r']}, "
        f"perm p={s['p2_valence_organization_perm_p']})"
    )

[plain (paper's exact format)] P1 PASS (positive 1002 vs negative 1002); P2 FAIL (max |r|=0.4141 bar 0.5, organization r=-0.2293, perm p=0.0034)


[chat template] P1 PASS (positive 1727 vs negative -578); P2 PASS (max |r|=0.7013 bar 0.5, organization r=0.4189, perm p=0.0)


<details><summary><b>How to read these panels</b></summary>

Left panel (the paper's sorted bar chart): each bar is one of the 164 NRC-matched emotion probes;
its height is the Pearson correlation between that probe's activation on the activity tokens of
"How would you feel about {activity}?" and the activity's Elo. In the paper this is strongly
valence-organized (hostile at -0.74 to blissful at +0.71). Right panel (the paper's scatter):
the single best probe against Elo, one dot per activity, colored by the 8 activity categories.
In the paper unsafe/misaligned activities sit at the bottom (Elo ~583) and helpful/engaging at
the top (~2465).

Registered reads (declared in scripts/score_preferences.py before collection landed): P1, positive
categories out-Elo negative ones; P2, max |r| at least 0.5 with permutation-significant positive
valence organization. The verdict line under each figure reports both. Elo scale note: our anchor
is mean 1000 (the paper's anchor is unpublished), so only gaps and rankings are comparable.

</details>

## 4. Steering (paper Figure 4, right half): the causal test

If the vectors carry causal preference content, adding an emotion vector to the residual
stream during the A/B choice should shift the resulting Elo, and the shift should track the
correlational profile from section 3 (the paper reports r = 0.85 between the two, with mean
shifts of +212 for blissful and -303 for hostile steering). We steered all 12 battery
emotions at layer 33 during the full chat-format pair pass, at two doses spanning the
coherence-preserving range (TREE Q1.H3.E2; the dose escalation exists because the alpha
calibration statistic turned out to be fit-noise-dominated, documented in the tree).

In [3]:
# this cell renders the steering arm (paper Figure 4 bottom scatter): redistribution vs probe-Elo r, both doses
STEER_ARMS = [
    ("alpha=2", ROOT / "results/steering_it_fixed"),
    ("alpha=8", ROOT / "results/steering_it_a8_fixed"),
]

fig = go.Figure()
for arm_label, arm_dir in STEER_ARMS:
    s = json.loads((arm_dir / "scores.json").read_text())
    fig.add_scatter(
        x=[r["probe_elo_r"] for r in s["per_emotion"]],
        y=[r["mean_delta_elo_positive_categories"] for r in s["per_emotion"]],
        mode="markers+text",
        text=[r["emotion"] for r in s["per_emotion"]],
        textposition="top center",
        textfont=dict(size=9),
        name=f"{arm_label} (P2 r={s['p2_pearson_r']:+.2f})",
    )
fig.add_hline(y=0, line_color="gray", line_width=1)
fig.add_vline(x=0, line_color="gray", line_width=1)
fig.update_layout(
    title=(
        "Steering moves preferences in the valence-correct direction, gemma-4-31b-it, layer 33"
        "<br><sup>maps to: Anthropic Figure 4, bottom scatter (their r=0.85, mean deltas +212/-303; "
        "ours: valence-sign test 11/12 and 10/12, dose-responsive)</sup>"
    ),
    xaxis_title="probe-Elo correlation r (unsteered, from the chat arm)",
    yaxis_title="mean delta Elo, positive categories (steered - baseline)",
    height=480,
    margin=dict(t=80, b=10),
)
fig.show()
for arm_label, arm_dir in STEER_ARMS:
    s = json.loads((arm_dir / "scores.json").read_text())
    print(
        f"[{arm_label}] P1 {'PASS' if s['p1_pass'] else 'FAIL'}; "
        f"P2 {'PASS' if s['p2_pass'] else 'FAIL'} (r={s['p2_pearson_r']:+.3f}, p={s['p2_p_value']:.3f}); "
        f"P3 {'PASS' if s['p3_pass'] else 'FAIL'} ({s['p3_sign_agreements']}/12)"
    )

[alpha=2] P1 PASS; P2 FAIL (r=+0.322, p=0.307); P3 PASS (11/12)
[alpha=8] P1 PASS; P2 FAIL (r=+0.228, p=0.475); P3 PASS (10/12)


<details><summary><b>How to read this scatter</b></summary>

One dot per steered emotion, at both doses. The x axis is how well that emotion's probe
*predicted* preferences without steering (section 3's per-probe r). The y axis is how much
steering with that vector actually *moved* preferences, measured as the mean Elo change of
the positive-category activities (the identifiable redistribution statistic: with a
mean-anchored Elo, a uniform shift of all activities is invisible to pairwise choices by
construction, which is also why the paper's +212/-303 global means need their unpublished
anchoring convention to interpret). In the paper this scatter has slope: emotions that
predict preference also drive it (r = 0.85). Ours is flat at both doses (r = -0.03 and
+0.15, p > 0.6), with redistribution never exceeding 7 Elo — about 2% of the paper's effect —
while the model's choice behavior stays fully coherent (mean logit-gap ratios 0.85-1.10, so
the null is not a broken-model artifact).

Registered reads (TREE Q1.H3.E2, amended for the anchor degeneracy before scoring): P1
coherence, P2 correlation at bar 0.5, P3 valence-sign agreement at 9/12. Verdict: the causal
null is dose-robust across the coherence-preserving range. Scope limits: single steering
layer (33), all-position additive steering.

</details>